# 🔎 MEF · Explorar la fuente del canon petrolero por municipalidad (2003–2024)

PERUPETRO publica el canon por municipalidad solo para 1993–2002 (Excel de Distribución) y el año en curso (Excel de Canon Municipalidades). Para completar **2003–2024** se busca la fuente del **MEF** (transferencias a gobiernos locales / ingresos por canon y sobrecanon petrolero).

Este notebook **no descarga datos**: solo revisa qué páginas del MEF responden desde Colab y qué filtros ofrecen (formularios, listas, enlaces con «canon» o «transferencia»). Mándame capturas de la salida para armar el lector.

- Si conoces la página exacta (la abres en el navegador: Transparencia Económica → Transferencias…), pégala en `URL_EXTRA`.
- Las direcciones de la lista son **candidatas**: algunas pueden no existir o haber cambiado.

In [ ]:
#@title Explorar páginas del MEF { display-mode: "form" }
URL_EXTRA = ""  #@param {type:"string"}
import re, os, time, requests, html
from urllib.parse import urljoin
CANDIDATAS = [
    "https://apps5.mineco.gob.pe/transferencias/gl/default.aspx",
    "https://apps5.mineco.gob.pe/transferencias/gl/",
    "https://apps5.mineco.gob.pe/transparencia/Ingresos/",
    "https://apps5.mineco.gob.pe/transparencia/mensual/",
    "https://www.mef.gob.pe/es/transferencias-a-los-gobiernos-nacional-regional-y-local",
]
if URL_EXTRA.strip():
    CANDIDATAS.insert(0, URL_EXTRA.strip())
S = requests.Session()
S.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
                  "Accept-Language": "es-PE,es;q=0.9"})
os.makedirs("/content/mef_explora", exist_ok=True)
def texto(x):
    return html.unescape(re.sub(r"<[^>]+>", " ", x)).strip()
for i, url in enumerate(CANDIDATAS, 1):
    print("=" * 100); print(f"[{i}] {url}")
    try:
        r = S.get(url, timeout=40, allow_redirects=True)
    except Exception as e:
        print(f"   ❌ sin respuesta: {type(e).__name__}: {str(e)[:150]}"); continue
    h = r.text or ""
    tit = re.search(r"<title[^>]*>(.*?)</title>", h, re.S | re.I)
    print(f"   HTTP {r.status_code} · {len(h):,} caracteres · URL final: {r.url}")
    print(f"   Título: {texto(tit.group(1))[:120] if tit else '–'}")
    print(f"   Menciones: canon={len(re.findall('canon', h, re.I))} · petrolero={len(re.findall('petroler', h, re.I))} · "
          f"transferencia={len(re.findall('transferen', h, re.I))} · incapsula={'sí' if 'incapsula' in h.lower() else 'no'}")
    open(f"/content/mef_explora/pagina_{i}.html", "w", encoding="utf-8").write(h)
    for f in re.findall(r"<form[^>]*>", h, re.I)[:3]:
        print("   FORM:", f[:160])
    for nombre, cuerpo in re.findall(r"<select[^>]*name=[\"']([^\"']+)[\"'][^>]*>(.*?)</select>", h, re.S | re.I)[:12]:
        ops = [texto(o) for o in re.findall(r"<option[^>]*>(.*?)</option>", cuerpo, re.S | re.I)]
        print(f"   LISTA {nombre} ({len(ops)} opciones): {' | '.join(ops[:12])[:300]}")
    links = [(urljoin(r.url, a), texto(t)) for a, t in re.findall(r"<a[^>]*href=[\"']([^\"'#]+)[\"'][^>]*>(.*?)</a>", h, re.S | re.I)]
    utiles = [(a, t) for a, t in links if re.search(r"canon|transfer|petrol|ingreso", a + " " + t, re.I)]
    for a, t in utiles[:25]:
        print(f"   ENLACE: {t[:70]!r} → {a[:140]}")
    time.sleep(2)
print("\nHTML guardado en /content/mef_explora (panel Archivos de Colab), por si hace falta revisarlo.")
